# R2 — Multi-model Experiment

**Scope**: 2 elite arms × N GPT models × 200 questions = 2N × 200 inferences

**Arms**: `elite_no_retrieval`, `elite_graphrag`

**Models**: configurable list (gpt-4.1, gpt-4o, gpt-5-mini, ...)

**2D Parallelism**: vừa parallel arms vừa parallel models. Với 3 models → 6 combos chạy concurrent (cần T4 16GB VRAM).

**Wall time on Colab Pro T4** với 3 models:
- Sequential: ~10-12h
- 6 combos parallel: ~3-4h (gpt-5-mini reasoning là long-pole)

**Supported metrics** (17 total): citation_validity, citation_recall/precision, faithfulness, content_hallucination_rate, invented_citation_rate, answer_relevance, bertscore_f1, cost_usd, latency_s, prolog_success_rate (+ 3 prolog), pairwise_consensus (GR vs NR per model), api_error_rate, McNemar/Bootstrap/Bonferroni significance.

## Prerequisites
1. Code repo trên GDrive: `/MyDrive/legal-graph-kb`
2. Neo4j Aura + secrets: `OPENAI_API_KEY`, `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`
3. OpenAI Tier 4+ recommended cho 6+ concurrent RPM

## ⚙️ CONFIG

In [ ]:
# ═══ MODELS ═══
# List models để compare. OpenAI public: gpt-4.1, gpt-4o, gpt-5-mini, gpt-5, gpt-4o-mini, o3-mini
R2_MODELS      = ['gpt-4.1', 'gpt-4o', 'gpt-5-mini']

# Judge model cho compute_metrics (LLM-as-Judge: faithfulness, halluc, pairwise, ...)
JUDGE_MODEL    = 'gpt-4o-mini'   # Khuyến nghị consistent

# ═══ ARMS ═══
# R2 chỉ so sánh 2 elite variants (no retrieval vs graphrag retrieval)
ARMS           = ['elite_no_retrieval', 'elite_graphrag']

# ═══ SCOPE ═══
N_QUESTIONS    = 200             # max 200 (set 10 cho pilot)
BACKFILL_PLAIN = True            # backfill plain_answer cho elite (~$0.50 với 6 combos)

# ═══ PARALLELISM ═══
# 2D: ARMS × MODELS. Total combos = len(ARMS) × len(R2_MODELS)
# Cap concurrent theo T4 VRAM + OpenAI RPM tier
MAX_PARALLEL   = 6               # OpenAI Tier 4+ ok với 6 concurrent

# ═══ PATHS ═══
REPO_GDRIVE    = '/content/drive/MyDrive/legal-graph-kb'
RESULTS_GDRIVE = '/content/drive/MyDrive/legal-graph-kb-results'

print(f'R2 models:    {R2_MODELS}')
print(f'R2 arms:      {ARMS}')
print(f'Judge model:  {JUDGE_MODEL}')
print(f'Total combos: {len(ARMS)} × {len(R2_MODELS)} = {len(ARMS)*len(R2_MODELS)}')
print(f'Total inferences: {len(ARMS)*len(R2_MODELS)*N_QUESTIONS}')

# Phase 1 — Setup

## 1.1 GPU + GDrive + secrets

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

from google.colab import drive, userdata
import os, sys, subprocess, time, json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
drive.mount('/content/drive')

for k in ['OPENAI_API_KEY', 'NEO4J_URI', 'NEO4J_USER', 'NEO4J_PASSWORD']:
    os.environ[k] = userdata.get(k) or ''
os.environ['NEO4J_DATABASE'] = 'neo4j'
os.environ['EMBED_DEVICE'] = 'cuda'
os.environ['PYTHONIOENCODING'] = 'utf-8'
print({k: ('set' if os.environ.get(k) else 'MISSING')
       for k in ['OPENAI_API_KEY', 'NEO4J_URI', 'NEO4J_USER', 'NEO4J_PASSWORD']})

## 1.2 Copy repo + install deps + symlink

In [ ]:
REPO_DIR = '/content/legal-graph-kb'
if not os.path.exists(REPO_DIR):
    !cp -r $REPO_GDRIVE $REPO_DIR
%cd $REPO_DIR
sys.path.insert(0, REPO_DIR)

!pip install -q neo4j sentence-transformers openai python-dotenv bert-score tqdm 2>&1 | tail -3
!apt-get install -y -q swi-prolog 2>&1 | tail -2

os.makedirs(f'{RESULTS_GDRIVE}/data/eval', exist_ok=True)
if os.path.islink('data/eval') or not os.path.exists('data/eval'):
    !rm -rf data/eval && ln -s $RESULTS_GDRIVE/data/eval data/eval
!ls -la data/eval | head -3

## 1.3 Neo4j Aura verify + push KG

In [ ]:
from neo4j import GraphDatabase
driver = GraphDatabase.driver(os.environ['NEO4J_URI'],
    auth=(os.environ['NEO4J_USER'], os.environ['NEO4J_PASSWORD']))
with driver.session(database='neo4j') as s:
    n_art = s.run('MATCH (n:Article) RETURN count(n) AS c').single()['c']
    print(f'Aura: {n_art} Articles')
    has_vec = any('clause_vec' in (i.get('name') or '')
                   for i in s.run('SHOW INDEXES').data())
    if not has_vec:
        s.run('''CREATE VECTOR INDEX clause_vec IF NOT EXISTS
                 FOR (c:Clause) ON c.embedding
                 OPTIONS {indexConfig: {`vector.dimensions`: 1024,
                                         `vector.similarity_function`: 'cosine'}}''')
driver.close()
if n_art == 0:
    !python -m src.load_neo4j 2>&1 | tail -10

## 1.4 Smoke test — 1 question × mỗi model

In [ ]:
from experiments.elite_pipelines import EliteGraphRAGPipeline
Q = 'Tôi đã đóng BHXH 12 năm, có đủ điều kiện hưởng lương hưu không?'
for m in R2_MODELS:
    print(f'\n=== Smoke: elite_graphrag × {m} ===')
    ep = EliteGraphRAGPipeline(model=m)
    ans = ep.ask(Q)
    print(f'  prolog_success: {ans.prolog_success}, elapsed: {ans.elapsed_s}s')
    print(f'  plain_answer: {(ans.plain_answer or "(empty)")[:200]}')
    ep.close()

# Phase 2 — Inference (2D parallel: ARMS × MODELS)

All `len(ARMS) × len(R2_MODELS)` combos concurrent.

In [ ]:
os.makedirs('logs', exist_ok=True)

def run_combo(cmd, name, env_overrides=None):
    t0 = time.time()
    env = os.environ.copy()
    if env_overrides: env.update(env_overrides)
    with open(f'logs/{name}.log', 'w', encoding='utf-8') as f:
        proc = subprocess.run(cmd, shell=True, stdout=f,
                              stderr=subprocess.STDOUT, env=env)
    return name, proc.returncode, time.time() - t0

def parallel(cmds_with_env, max_workers):
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = {ex.submit(run_combo, c, n, e): n for c, n, e in cmds_with_env}
        for fut in as_completed(futs):
            n, code, dur = fut.result()
            status = '✓' if code == 0 else f'✗ exit={code}'
            print(f'[{time.strftime("%H:%M:%S")}] {status} {n} ({dur:.0f}s)')
    print(f'\nWall time: {(time.time()-t0)/60:.1f} min')

# Build 2D cmd list: ARMS × R2_MODELS — flatten thành 1 list
cmds = []
for arm in ARMS:
    for model in R2_MODELS:
        cmds.append((
            f'python -m experiments.run_multimodel_inference '
            f'--arms {arm} --models {model} --n {N_QUESTIONS}',
            f'r2_{arm}_{model.replace(".","_")}',
            None,  # --models flag inline, no env needed
        ))

print(f'Launching {len(cmds)} combos (= {len(ARMS)} arms × {len(R2_MODELS)} models)')
print(f'parallel max {MAX_PARALLEL} concurrent...\n')
parallel(cmds, max_workers=MAX_PARALLEL)

## 2.x Progress check — per-combo file count + API errors

In [ ]:
root = Path('data/eval/multimodel/results')
if not root.exists():
    print('No R2 results yet')
else:
    print(f'{"combo":<40} {"done":>6} {"api_err":>8}')
    print('-'*60)
    for d in sorted(root.iterdir()):
        if not d.is_dir(): continue
        n = len([f for f in d.glob('A*.json') if not f.name.endswith('.error.json')])
        n_api = sum(1 for f in d.glob('A*.json')
                    if not f.name.endswith('.error.json')
                    and json.loads(f.read_text(encoding='utf-8')).get('prompt_tokens') == 0)
        print(f'  {d.name:<38} {n:>3}/{N_QUESTIONS}  {n_api:>3}')

# Phase 3 — Metrics + Report

## 3.1 Backfill plain_answer (R2 elite combos)

In [ ]:
if BACKFILL_PLAIN:
    os.environ['BACKFILL_MODEL'] = 'gpt-4o-mini'
    # Build combos list cho R2 only
    r2_combos = ','.join(
        f'{arm}__{m.replace(".","_")}'
        for arm in ARMS for m in R2_MODELS
    )
    !python -m experiments.rerender_plain_answer --combos $r2_combos 2>&1 | tail -20
else:
    print('Skipped')

## 3.2 Compute R2 metrics (judge = JUDGE_MODEL)

In [ ]:
env_judge = os.environ.copy()
env_judge['OPENAI_MODEL'] = JUDGE_MODEL
subprocess.run('python -m experiments.compute_multimodel_metrics',
               shell=True, env=env_judge)

## 3.3 Audit fixes + report + significance

In [ ]:
!python -m experiments.audit_apply_fixes 2>&1 | tail -8
!python -m experiments.audit_apply_fixes_v2 2>&1 | tail -8
!python -m experiments.generate_multimodel_report 2>&1 | tail -3
!python -m experiments.compute_significance 2>&1 | tail -3

# Phase 4 — Visualizations 📈 (R2 — multi-model)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

def ch(rec, *keys):
    v = rec
    for k in keys:
        if not isinstance(v, dict): return None
        v = v.get(k)
    return v

r2_data = json.loads(Path('data/eval/multimodel/metrics.json').read_text(encoding='utf-8'))
rows = []
for combo, recs in r2_data.items():
    for r in recs:
        if r.get('api_error'): continue
        ps = ch(r, 'prolog_rollback', 'prolog_success')
        fs = ch(r, 'prolog_rollback', 'first_try_success')
        rows.append({
            'arm': r.get('arm', combo.split('__')[0]),
            'model': r.get('model', ''),
            'combo': combo,
            'stt': r.get('stt'),
            'citation_validity': ch(r, 'citation_validity', 'validity_rate'),
            'citation_recall': ch(r, 'citation_recall', 'recall'),
            'citation_precision': ch(r, 'citation_precision', 'precision'),
            'faithfulness': ch(r, 'faithfulness', 'faithfulness'),
            'content_hallu': ch(r, 'hallucination', 'content_hallucination_rate'),
            'invented_cit': ch(r, 'hallucination', 'invented_citation_rate'),
            'answer_relevance': ch(r, 'answer_relevance', 'answer_relevance'),
            'bertscore_f1': ch(r, 'bertscore', 'bertscore_f1'),
            'cost_usd': ch(r, 'cost', 'cost_usd'),
            'latency_s': ch(r, 'latency', 'latency_s'),
            'prolog_success': 1.0 if ps else 0.0 if ps is not None else None,
            'first_try_success': 1.0 if fs else 0.0 if fs is not None else None,
        })
df = pd.DataFrame(rows)
print(f'R2 clean records: {len(df)}')
df.head()

## 4.1 Grouped bar — Prolog success per (arm, model)

In [ ]:
agg = df.groupby(['model', 'arm']).agg(
    prolog_success_rate=('prolog_success', 'mean'),
    n=('prolog_success', 'count'),
).reset_index()
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=agg, x='model', y='prolog_success_rate',
             hue='arm', ax=ax, palette='Set2')
ax.set_title('R2 Prolog Success Rate — model × arm (API errors excluded)')
ax.set_ylim(0, 1.05); ax.set_ylabel('success rate')
ax.legend(title='Arm', loc='lower right')
for p in ax.patches:
    h = p.get_height()
    if h > 0:
        ax.text(p.get_x() + p.get_width()/2, h + 0.01,
                 f'{h:.2f}', ha='center', fontsize=8)
plt.tight_layout(); plt.show()

## 4.2 Heatmap — Metric × (model, arm) full matrix

In [ ]:
METRIC_COLS = ['citation_validity', 'citation_recall', 'citation_precision',
                'faithfulness', 'content_hallu', 'invented_cit',
                'answer_relevance', 'bertscore_f1', 'prolog_success']
df['model_arm'] = df['model'] + ' / ' + df['arm']
heat = df.groupby('model_arm')[METRIC_COLS].mean()
fig, ax = plt.subplots(figsize=(12, max(4, 0.5*len(heat))))
sns.heatmap(heat, annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=0, vmax=1, cbar_kws={'label': 'mean'}, ax=ax)
ax.set_title('R2 Metric Matrix — mean per (model, arm)')
ax.set_xlabel(''); ax.set_ylabel('')
plt.tight_layout(); plt.show()

## 4.3 Faithfulness diff (GR − NR) per model

In [ ]:
diffs = []
for model in R2_MODELS:
    nr = df[(df['model']==model) & (df['arm']=='elite_no_retrieval')]['faithfulness'].mean()
    gr = df[(df['model']==model) & (df['arm']=='elite_graphrag')]['faithfulness'].mean()
    if pd.notna(nr) and pd.notna(gr):
        diffs.append({'model': model, 'GR − NR': gr - nr, 'NR': nr, 'GR': gr})
df_diff = pd.DataFrame(diffs)
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(df_diff['model'], df_diff['GR − NR'],
               color=['#2ca02c' if v >= 0 else '#d62728' for v in df_diff['GR − NR']])
ax.axhline(0, color='gray', lw=0.5)
ax.set_ylabel('Δ faithfulness (GR − NR)')
ax.set_title('R2: Does GraphRAG help faithfulness vs no-retrieval?')
for b, v in zip(bars, df_diff['GR − NR']):
    ax.text(b.get_x() + b.get_width()/2,
             v + (0.005 if v >= 0 else -0.012),
             f'{v:+.3f}', ha='center', fontsize=9)
plt.tight_layout(); plt.show()

## 4.4 Cost vs Faithfulness scatter — per (model, arm)

In [ ]:
agg = df.groupby(['model', 'arm']).agg(
    cost_usd=('cost_usd', 'mean'),
    faithfulness=('faithfulness', 'mean'),
    latency_s=('latency_s', 'mean'),
).reset_index().dropna()
fig, ax = plt.subplots(figsize=(10, 6))
for arm in ARMS:
    sub = agg[agg['arm'] == arm]
    marker = 'o' if arm == 'elite_graphrag' else 's'
    ax.scatter(sub['cost_usd'], sub['faithfulness'],
               s=sub['latency_s']*5, alpha=0.65, label=arm, marker=marker)
    for _, row in sub.iterrows():
        ax.annotate(row['model'], (row['cost_usd'], row['faithfulness']),
                    fontsize=8, xytext=(5,5), textcoords='offset points')
ax.set_xlabel('Mean cost (USD per question)')
ax.set_ylabel('Mean faithfulness')
ax.set_title('R2 Cost vs Faithfulness (bubble = latency, marker = arm)')
ax.set_xscale('log')
ax.legend()
plt.tight_layout(); plt.show()

## 4.5 Pairwise consensus — GR vs NR per model

In [ ]:
from collections import Counter
rows_pw = []
for combo, recs in r2_data.items():
    if not combo.startswith('elite_graphrag__'): continue
    model = combo.replace('elite_graphrag__', '').replace('_', '.')
    for r in recs:
        if r.get('api_error'): continue
        pw = r.get('pairwise_vs_no_retrieval')
        if pw:
            c = pw['consensus']
            # Normalize labels cho human readability
            if 'graphrag' in c: lab = 'elite_graphrag wins'
            elif 'no_retrieval' in c: lab = 'elite_no_retrieval wins'
            else: lab = c
            rows_pw.append({'model': model, 'consensus': lab})
if rows_pw:
    df_pw = pd.DataFrame(rows_pw)
    pivot = df_pw.groupby(['model', 'consensus']).size().unstack(fill_value=0)
    pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
    fig, ax = plt.subplots(figsize=(10, 4))
    pct.plot(kind='barh', stacked=True, ax=ax,
              color=['#2ca02c', '#d62728', '#ff7f0e', '#1f77b4'])
    ax.set_xlabel('% of records')
    ax.set_title('R2 Pairwise: GR vs NR — % wins per model (clean records)')
    ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=9)
    plt.tight_layout(); plt.show()

## 4.6 Box plot — Latency per (arm, model)

In [ ]:
df_plot = df.dropna(subset=['latency_s']).copy()
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=df_plot, x='model', y='latency_s', hue='arm',
             ax=ax, palette='Set2')
ax.set_yscale('log')
ax.set_title('R2 Latency Distribution — model × arm (log scale)')
ax.set_ylabel('seconds')
plt.tight_layout(); plt.show()

## 4.7 Sortable summary DataFrame

In [ ]:
summary = df.groupby(['model', 'arm']).agg(
    n=('stt', 'count'),
    cit_valid=('citation_validity', 'mean'),
    cit_recall=('citation_recall', 'mean'),
    cit_precision=('citation_precision', 'mean'),
    faithfulness=('faithfulness', 'mean'),
    content_hallu=('content_hallu', 'mean'),
    invented_cit=('invented_cit', 'mean'),
    bertscore=('bertscore_f1', 'mean'),
    cost_usd=('cost_usd', 'mean'),
    latency_s=('latency_s', 'mean'),
    prolog_success=('prolog_success', 'mean'),
    first_try=('first_try_success', 'mean'),
).round(4)
summary.style.background_gradient(cmap='RdYlGn', axis=0,
    subset=['cit_valid','cit_recall','cit_precision','faithfulness','bertscore',
            'prolog_success','first_try'])\
    .background_gradient(cmap='RdYlGn_r', axis=0,
    subset=['content_hallu','invented_cit','cost_usd','latency_s'])

## 4.8 Display reports + sync + download

In [ ]:
from IPython.display import Markdown, display
for f in ['reports/multimodel_report.md', 'reports/significance.md']:
    if os.path.exists(f):
        print(f'\n{"="*70}\n{f}\n{"="*70}')
        display(Markdown(open(f, encoding='utf-8').read()[:5000]))

!mkdir -p $RESULTS_GDRIVE/reports
!rsync -av reports/ $RESULTS_GDRIVE/reports/ 2>&1 | tail -3
!cd /content/legal-graph-kb && zip -rq /content/r2_bundle.zip \
    reports/ data/eval/multimodel/metrics.json data/eval/multimodel/metrics.csv
from google.colab import files
files.download('/content/r2_bundle.zip')
print(f'\n✓ R2 pipeline complete ({len(R2_MODELS)} models × {len(ARMS)} arms)')